# Divisibility Rules: The Common Skeleton



> *You've derived 12 divisibility rules individually. They all follow the same template. Here's the one thing that ties them together.*

---



## 1. The Algebraic Skeleton You Already Used



Every single divisibility notebook in this repo follows the same steps. Let's make it explicit:



**Step 1**: Write the number in base-10:



$$

N = d_{n-1} \cdot 10^{n-1} + d_{n-2} \cdot 10^{n-2} + \cdots + d_1 \cdot 10 + d_0

$$



**Step 2**: For the divisor $d$, express each power of 10 as a multiple of $d$ plus a remainder:



$$

10^i = q_i \cdot d + r_i \qquad \text{where } 0 \le r_i < d

$$



**Step 3**: Substitute back:



$$

\begin{align}

N &= \sum_{i} d_i \cdot (q_i \cdot d + r_i) \\

  &= d \cdot \underbrace{\sum_i d_i q_i}_{\text{multiple of } d} + \underbrace{\sum_i d_i r_i}_{\text{the remainder}}

\end{align}

$$



**Step 4**: The rule drops out:



$$

\boxed{N \text{ is divisible by } d \iff \sum_{i=0}^{n-1} d_i \cdot r_i \equiv 0 \pmod{d}}

$$



That's it. **Every** divisibility rule in your repo is this single formula — with $r_i = 10^i \bmod d$ as the only moving part.



The different rules exist because $r_i$ behaves differently for different $d$.

---



## 2. The Three Families (Explained by $10^i \bmod d$)



The remainder sequence $r_i = 10^i \bmod d$ determines everything. There are only three patterns it can follow:



### Family A: Last N Digits



**When does it happen?** When $d$ divides $10^t$ for some $t$. Then $r_i = 0$ for all $i \ge t$, and only the last $t$ digits matter.



Since $10^t = 2^t \cdot 5^t$, this happens when $d$ has only 2 and 5 as prime factors:



| Divisor | Because | Rule: check last... |

|---|---|---|

| $2$ | $10^1 \div 2 = 5$, $r=0$ | 1 digit (ones place) |

| $4 = 2^2$ | $10^2 \div 4 = 25$, $r=0$ | 2 digits |

| $5$ | $10^1 \div 5 = 2$, $r=0$ | 1 digit (ones place) |

| $8 = 2^3$ | $10^3 \div 8 = 125$, $r=0$ | 3 digits |

| $10$ | $10^1 \div 10 = 1$, $r=0$ | 1 digit |

| $2^n$ | $10^n \div 2^n = 5^n$, $r=0$ | $n$ digits |

| $5^n$ | $10^n \div 5^n = 2^n$, $r=0$ | $n$ digits |



The remainder sequence is: $r_0, r_1, r_2, \dots, r_{t-1}$, then all zeros.



### Family B: Digit Sum



**When does it happen?** When $10 \equiv 1 \pmod{d}$. Then $10^i \equiv 1 \pmod{d}$ for ALL $i$, so $r_i = 1$ always.



This means $10 - 1 = 9$ is divisible by $d$, so $d$ divides 9:



| Divisor | Because | Rule |

|---|---|---|

| $3$ | $10 \equiv 1 \pmod{3}$ | Sum of digits $\equiv 0 \pmod{3}$ |

| $9$ | $10 \equiv 1 \pmod{9}$ | Sum of digits $\equiv 0 \pmod{9}$ |



Why does recursive summing work? Because when you get a multi-digit sum, you can apply the same rule again — the digit-sum operation preserves congruence modulo 3 or 9.



### Family C: Periodic Remainders (The General Case)



**When does it happen?** For all other $d$. The sequence $r_i = 10^i \bmod d$ follows a repeating cycle.



The cycle length is the **order** of 10 modulo $d$ (the smallest $t$ such that $10^t \equiv 1 \pmod{d}$).



| Divisor | $10^i \bmod d$ cycle | Rule form |

|---|---|---|

| $7$ | $3, 2, 6, 4, 5, 1$ (length 6) | Weighted sum with $3,2,6,4,5,1$ cycle |

| $11$ | $10, 1$ (length 2) | Alternating sum of digits |

| $13$ | $10, 9, 12, 3, 4, 1$ (length 6) | Weighted sum with $10,9,12,3,4,1$ cycle |

| $6$ | — | Composite: check 2 AND 3 |

| $12$ | — | Composite: check 3 AND 4 |



The "chop-off" rules you derived for 7, 11, 13 are clever **shortcuts** — they avoid tracking the full cycle by progressively reducing the number one digit at a time.

---



## 3. Walking Through It with a Concrete Number



Let's take $N = 3829$ and test divisibility by $7$ using the raw $10^i \bmod 7$ method:



$$

\begin{align}

3829 &= 3\cdot10^3 + 8\cdot10^2 + 2\cdot10 + 9 \\

10^0 \bmod 7 &= 1 \\

10^1 \bmod 7 &= 3 \\

10^2 \bmod 7 &= 2 \\

10^3 \bmod 7 &= 6 \\

\end{align}

$$



$$

3829 \bmod 7 = (3\cdot6 + 8\cdot2 + 2\cdot3 + 9\cdot1) \bmod 7 = (18 + 16 + 6 + 9) \bmod 7 = 49 \bmod 7 = 0

$$



So 3829 is divisible by 7. ($3829 = 7 \cdot 547$)



Your chop-off rule ($\text{rest} - 2 \cdot \text{ones}$) is doing the same calculation in a more clever way — it avoids multiplying each digit by a different weight, reducing the problem to a smaller number each time.

---



## 4. One Function to Rule Them All



The template is so general that a single Python function can generate any divisibility rule.

In [ ]:
def divisible_by(N, d):

    """

    Check if N is divisible by d using the modular arithmetic template.

    

    Computes N mod d by weighting each digit by 10^i mod d.

    """

    # Precompute remainders of powers of 10 modulo d

    # (We only need up to the number of digits in N)

    powers_of_10_mod_d = []

    current = 1  # 10^0 mod d = 1

    for i in range(len(str(N))):

        powers_of_10_mod_d.append(current)

        current = (current * 10) % d

    

    # Extract digits and compute weighted sum

    digits = [int(ch) for ch in str(N)][::-1]  # ones place first

    remainder = sum(digit * r for digit, r in zip(digits, powers_of_10_mod_d)) % d

    

    return remainder == 0, remainder



# Test against Python's built-in modulo

test_numbers = [7, 12, 123, 4567, 89123, 999999, 1234567]

test_divisors = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]



print("Testing the general template against Python's % operator...")

all_ok = True

for N in test_numbers:

    for d in test_divisors:

        result, _ = divisible_by(N, d)

        expected = (N % d == 0)

        if result != expected:

            print(f"  MISMATCH: {N} % {d}: got {result}, expected {expected}")

            all_ok = False

if all_ok:

    print(f"  All {len(test_numbers) * len(test_divisors)} tests passed ✓")



print()

print("Example: checking 3829 for divisibility by each divisor:")

for d in test_divisors:

    ok, rem = divisible_by(3829, d)

    print(f"  3829 % {d:2d}: remainder = {rem}, {'DIVISIBLE ✓' if ok else 'not divisible'}")

---



## 5. The Remainder Table



The entire divisibility rule for any divisor $d$ is encoded in this table: the sequence $10^i \bmod d$.

In [ ]:
def remainder_cycle(d, max_len=12):

    """Return the sequence of 10^i mod d for i = 0..max_len-1."""

    seq = []

    current = 1

    for i in range(max_len):

        seq.append(current)

        current = (current * 10) % d

    return seq



print("Remainder sequences 10^i mod d for each divisor d:")

print("=" * 70)

print(f"{'d':>3} | {'Family':<14} | 10^i mod d sequence")

print("-" * 70)



for d in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]:

    seq = remainder_cycle(d, 12)

    

    # Determine family

    if all(r == 0 for r in seq[1:]):  # r_1 onward all zero

        family = "Last 1 digit"

        for t in range(2, 6):

            if all(r == 0 for r in seq[t:]):

                family = f"Last {t} digits"

    elif all(r == 1 for r in seq):

        family = "Digit sum"

    elif d in [6, 12]:

        family = "Composite"

    else:

        # Find cycle length

        seq_str = ','.join(map(str, seq))

        for cyc_len in range(1, 7):

            pattern = seq[:cyc_len]

            if seq == pattern * (12 // cyc_len):

                family = f"Periodic (len {cyc_len})"

                break

        else:

            family = "Periodic"

    

    seq_str = ', '.join(map(str, seq[:8]))

    if len(seq) > 8:

        seq_str += ', ...'

    print(f"{d:3d} | {family:<14} | {seq_str}")



print()

print("Notice:")

print("  • 2, 4, 5, 8, 10: remainders become 0 early → last N digits")

print("  • 3, 9: remainders are all 1 → digit sum")

print("  • 7, 11, 13: remainders cycle → periodic/chop-off rules")

print("  • 6, 12: composite (check both factors)")

---



## 6. Where Do the Chop-Off Rules Come From?



The chop-off rules for 7, 11, 13 are shortcuts. Instead of tracking the full remainder cycle, they reduce the number one digit at a time.



The trick: since $10 \equiv r \pmod{d}$ (where $r = 10 \bmod d$), we can write:



$$

N = 10 \cdot a + b \equiv r \cdot a + b \pmod{d}

$$



where $a$ is the number without the last digit, and $b$ is the last digit.



So instead of weighting each digit by $10^i \bmod d$, we repeatedly replace $N$ with $r \cdot a + b$ (or some equivalent form) until we get a small number.



**For 7:** $10 \equiv 3 \pmod{7}$, so $10a + b \equiv 3a + b \pmod{7}$.

The classic rule $a - 2b$ works because $3a + b \equiv 0 \iff a - 2b \equiv 0$ (multiply both sides by 5, since $3 \cdot 5 \equiv 1 \pmod{7}$).



**For 11:** $10 \equiv -1 \pmod{11}$, so $10a + b \equiv -a + b \pmod{11}$.

This means $a - b \equiv 0 \pmod{11}$ — the alternating sum rule emerges naturally.



**For 13:** $10 \equiv 10 \pmod{13}$. The rule $a + 4b$ works because $10 \cdot 4 \equiv 1 \pmod{13}$, so $a + 4b \equiv 0 \iff 10a + b \equiv 0$.

In [ ]:
def chop_off_rule(N, d):

    """

    Find a chop-off reduction rule for divisor d.

    We want a multiplier k such that N ≡ (rest - k·last_digit) or (rest + k·last_digit).

    """

    # 10 mod d

    r = 10 % d

    # Find k such that k*r ≡ 1 (mod d), i.e., k is the modular inverse of r

    for k in range(1, d):

        if (k * r) % d == 1:

            # Then: 10a + b ≡ 0 (mod d) iff a + k·b ≡ 0 (mod d)

            return f"rest + {k}·last"

    return "no simple chop-off rule"



print("Chop-off reduction rules for each divisor:")

print("-" * 40)

for d in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]:

    r = 10 % d

    if r == 0:

        rule = "last digit rule (10 ≡ 0)"

    elif r == 1:

        rule = "digit sum (10 ≡ 1)"

    else:

        rule = chop_off_rule(N=0, d=d)

    print(f"  d={d:2d}: 10 ≡ {r:2d} (mod {d:2d}) → {rule}")

---



## 7. What This All Means



The 12 divisibility rules you derived are **not 12 separate facts**. They are 12 expressions of one single idea:



> **A number's remainder modulo $d$ is the weighted sum of its digits, weighted by $10^i \bmod d$.**



The different "rule types" are just convenient shortcuts that arise from the structure of $10^i \bmod d$:



| If $10^i \bmod d$ ... | Then the rule is ... | Because ... |

|---|---|---|

| becomes $0$ after position $t-1$ | Check last $t$ digits | Higher places contribute nothing |

| is $1$ everywhere | Sum the digits | Every place contributes its digit equally |

| follows a repeating cycle | Use the weighted sum or a chop-off shortcut | The weights are just the cycle values |

| doesn't apply cleanly | Composite rule: check factors | $d = ab$ with $\gcd(a,b)=1$ |



**The meta-pattern is modular arithmetic.** The specific rules are just human-friendly shortcuts for what the modular arithmetic already tells us.